# Covertype experiments with MLflow

1. Read the raw data from `raw.covertype` (loaded by Postgres on first start).
2. Process it (cast, validate) and write it to `processed.covertype`.
3. Read `processed.covertype` back and draw a fixed stratified sample.
4. Train 22 hyperparameter combinations, one MLflow run each.
5. Register the best run as a new version of `covertype` and move `@champion`
   only if it beats the current champion on the same test rows.
6. Tell the inference API to reload when the champion changed.

In [1]:
import csv
import hashlib
import os
import time
from io import StringIO
from itertools import product

import mlflow
import pandas as pd
import requests
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sqlalchemy import create_engine, text

# MLflow reads MLFLOW_TRACKING_URI from the environment. No MinIO credentials
# here: model files go to the MLflow server, which stores them in MinIO.
engine = create_engine(os.environ["DATA_DB_URL"])
API_URL = os.environ["API_URL"]

EXPERIMENT_NAME = "covertype_tuning"
MODEL_NAME = "covertype"
CHAMPION_ALIAS = "champion"

SAMPLE_ROWS = 100_000
TEST_SIZE = 0.2
RANDOM_STATE = 42

NUMERIC_COLUMNS = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CATEGORICAL_COLUMNS = ["wilderness_area", "soil_type"]
FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
TARGET = "cover_type"

print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: http://mlflow:5000


## 1. Raw data

Every value is still TEXT, exactly as in the CSV. `row_id` keeps the file order.

In [2]:
raw = pd.read_sql("SELECT * FROM raw.covertype ORDER BY row_id", engine)
print(f"{len(raw):,} raw rows")
raw.head()

581,012 raw rows


,row_id,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,cover_type,loaded_at
0,1,2596,51,3,258,0,510,221,232,148,6279,Rawah,C7745,4,2026-10-04 03:25:53.014183+00:00
1,2,2590,56,2,212,-6,390,220,235,151,6225,Rawah,C7745,4,2026-10-04 03:25:53.014183+00:00
2,3,2804,139,9,268,65,3180,234,238,135,6121,Rawah,C4744,1,2026-10-04 03:25:53.014183+00:00
3,4,2785,155,18,242,118,3090,238,238,122,6211,Rawah,C7746,1,2026-10-04 03:25:53.014183+00:00
4,5,2595,45,2,153,-1,391,220,234,150,6172,Rawah,C7745,4,2026-10-04 03:25:53.014183+00:00


## 2. Processing

Cast to numbers and drop invalid rows: non-numeric or non-integer values, empty
categories, aspect outside 0–360, hillshade outside 0–255. Same rules as the
`CHECK` constraints on `processed.covertype`.

In [3]:
processed = raw[["row_id"] + FEATURES + [TARGET]].copy()
for column in NUMERIC_COLUMNS + [TARGET]:
    processed[column] = pd.to_numeric(processed[column], errors="coerce")
for column in CATEGORICAL_COLUMNS:
    processed[column] = processed[column].str.strip()

whole_numbers = processed[NUMERIC_COLUMNS + [TARGET]]
valid = (
    whole_numbers.notna().all(axis=1)
    & (whole_numbers % 1 == 0).all(axis=1)
    & (processed[CATEGORICAL_COLUMNS].fillna("") != "").all(axis=1)
    & processed["aspect"].between(0, 360)
    & processed[["hillshade_9am", "hillshade_noon", "hillshade_3pm"]].apply(lambda s: s.between(0, 255)).all(axis=1)
)
processed = processed[valid].astype({column: "int64" for column in NUMERIC_COLUMNS + [TARGET]})
print(f"{len(processed):,} valid rows, {(~valid).sum():,} dropped")

581,012 valid rows, 0 dropped


## 3. Write `processed.covertype`

`TRUNCATE` + insert in one transaction: re-running the notebook replaces the
table instead of duplicating it. The insert uses Postgres `COPY`, much faster
than row-by-row `INSERT`s for 581k rows.

In [4]:
def copy_insert(table, conn, keys, data_iter):
    """pandas to_sql method that streams the rows through Postgres COPY."""
    buffer = StringIO()
    csv.writer(buffer).writerows(data_iter)
    buffer.seek(0)
    with conn.connection.cursor() as cursor:
        cursor.copy_expert(
            f"COPY {table.schema}.{table.name} ({', '.join(keys)}) FROM STDIN WITH CSV", buffer
        )


with engine.begin() as conn:
    conn.execute(text("TRUNCATE processed.covertype"))
    processed.to_sql("covertype", conn, schema="processed", if_exists="append",
                     index=False, method=copy_insert)

with engine.connect() as conn:
    print(f"{conn.execute(text('SELECT count(*) FROM processed.covertype')).scalar():,} rows in processed.covertype")

581,012 rows in processed.covertype


## 4. Training data, from `processed.covertype`

Training reads only the processed table. A fixed stratified sample of 100k rows
(80/20 split, `random_state=42`) keeps 22 runs fast and scores every run, and
every future champion comparison, on the same test rows. The fingerprint of the
test `row_id`s is logged to MLflow as proof.

In [5]:
data = pd.read_sql(
    f"SELECT row_id, {', '.join(FEATURES)}, {TARGET} FROM processed.covertype ORDER BY row_id", engine
)
sample, _ = train_test_split(data, train_size=SAMPLE_ROWS, stratify=data[TARGET], random_state=RANDOM_STATE)
train, test = train_test_split(sample, test_size=TEST_SIZE, stratify=sample[TARGET], random_state=RANDOM_STATE)

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
test_fingerprint = hashlib.md5(test["row_id"].sort_values().to_numpy().tobytes()).hexdigest()

print(f"train {len(train):,} rows, test {len(test):,} rows, test fingerprint {test_fingerprint}")
pd.DataFrame({
    "processed": data[TARGET].value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).sort_index().round(3)

train 80,000 rows, test 20,000 rows, test fingerprint e71d85d0ba560ef19310fbd98ddddba9


,processed,train,test
cover_type,,,
0,0.365,0.365,0.365
1,0.488,0.488,0.488
2,0.062,0.062,0.062
3,0.005,0.005,0.005
4,0.016,0.016,0.016
5,0.030,0.030,0.030
6,0.035,0.035,0.035


## 5. Candidates: 22 hyperparameter combinations

Each candidate is a full sklearn Pipeline (one-hot for the categories, scaling for
the numbers, then the classifier), so the model takes raw values like `"Rawah"`
directly. All classifiers use `class_weight="balanced"`: the classes are very
imbalanced.

In [6]:
def make_pipeline(classifier):
    preprocessor = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLUMNS),
        ("numeric", StandardScaler(), NUMERIC_COLUMNS),
    ])
    return Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


# model type -> (classifier factory, hyperparameter grid)
SEARCH_SPACE = {
    "random_forest": (
        lambda **params: RandomForestClassifier(class_weight="balanced", n_jobs=-1,
                                                random_state=RANDOM_STATE, **params),
        {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]},
    ),
    "hist_gradient_boosting": (
        lambda **params: HistGradientBoostingClassifier(class_weight="balanced",
                                                        random_state=RANDOM_STATE, **params),
        # Not max_iter: with >10k rows sklearn enables early stopping, so
        # training stops before 100 iterations and max_iter changes nothing.
        {"learning_rate": [0.05, 0.1, 0.2], "max_leaf_nodes": [15, 31, 63]},
    ),
    "logistic_regression": (
        lambda **params: LogisticRegression(class_weight="balanced", max_iter=2000, **params),
        {"C": [0.01, 0.1, 1, 10]},
    ),
}

# MLflow 3 saves sklearn models with skops (safer than pickle), which refuses
# tree internals unless they are explicitly trusted: a tampered tree file could
# crash predict(). We train these models ourselves, so we trust exactly the one
# type each family needs, found with skops.io.get_untrusted_types().
SKOPS_TRUSTED_TYPES = {
    "random_forest": ["sklearn.tree._tree.Tree"],
    "hist_gradient_boosting": ["sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor"],
    "logistic_regression": [],
}

candidates = [
    (model_type, dict(zip(grid, values)))
    for model_type, (_, grid) in SEARCH_SPACE.items()
    for values in product(*grid.values())
]
print(f"{len(candidates)} candidates")

22 candidates


## 6. Experiments: one MLflow run per candidate

All runs of one execution are grouped under a parent "sweep" run. Each child run
logs its hyperparameters, its test metrics (macro-F1, accuracy, F1 per class), the
fit time, and the model itself, with an input example from which MLflow infers
the model's signature.

In [7]:
def evaluate(model, X, y):
    predictions = model.predict(X)
    classes = sorted(y.unique())
    per_class = f1_score(y, predictions, average=None, labels=classes)
    return {
        "macro_f1": f1_score(y, predictions, average="macro"),
        "accuracy": accuracy_score(y, predictions),
        **{f"f1_class_{c}": score for c, score in zip(classes, per_class)},
    }


mlflow.set_experiment(EXPERIMENT_NAME)
sweep_name = time.strftime("sweep_%Y%m%dT%H%M%SZ", time.gmtime())
results = []

with mlflow.start_run(run_name=sweep_name) as sweep:
    mlflow.log_params({
        "data_source": "processed.covertype",
        "sample_rows": SAMPLE_ROWS, "train_rows": len(train), "test_rows": len(test),
        "random_state": RANDOM_STATE, "test_rows_fingerprint": test_fingerprint,
    })

    for number, (model_type, params) in enumerate(candidates, start=1):
        run_name = model_type + "_" + "_".join(f"{key}={value}" for key, value in params.items())
        build_classifier, _ = SEARCH_SPACE[model_type]

        with mlflow.start_run(run_name=run_name, nested=True) as run:
            model = make_pipeline(build_classifier(**params))
            started = time.perf_counter()
            model.fit(X_train, y_train)
            fit_seconds = time.perf_counter() - started
            metrics = evaluate(model, X_test, y_test)

            mlflow.set_tag("model_type", model_type)
            mlflow.log_params(params)
            mlflow.log_metrics({**metrics, "fit_seconds": fit_seconds})
            model_info = mlflow.sklearn.log_model(
                model, name="model", input_example=X_train.head(5),
                skops_trusted_types=SKOPS_TRUSTED_TYPES[model_type],
            )

        results.append({
            "run_name": run_name, "run_id": run.info.run_id, "model_uri": model_info.model_uri,
            "macro_f1": metrics["macro_f1"], "accuracy": metrics["accuracy"], "fit_seconds": fit_seconds,
        })
        print(f"[{number:2}/{len(candidates)}] {run_name:58} macro-F1 {metrics['macro_f1']:.3f}  ({fit_seconds:5.1f}s)")

    results = pd.DataFrame(results).sort_values("macro_f1", ascending=False, ignore_index=True)
    mlflow.log_metric("best_macro_f1", results.loc[0, "macro_f1"])
    mlflow.set_tag("best_run", results.loc[0, "run_name"])

results[["run_name", "macro_f1", "accuracy", "fit_seconds"]].round(3)

2026/10/04 19:37:36 INFO mlflow.tracking.fluent: Experiment with name 'covertype_tuning' does not exist. Creating a new experiment.


2026/10/04 19:37:36 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 1000'


2026/10/04 19:37:36 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/10/04 19:37:36 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git branch is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/10/04 19:37:36 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git repository URL is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



...

2026/10/04 19:43:18 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 1000'


/app/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/app/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference tim

2026/10/04 19:43:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run logistic_regression_C=10 at: http://mlflow:5000/#/experiments/1/runs/3c9660159e204df4adbb946a0fa62019
🧪 View experiment at: http://mlflow:5000/#/experiments/1
[22/22] logistic_regression_C=10                                   macro-F1 0.499  ( 16.8s)
🏃 View run sweep_20261004T193736Z at: http://mlflow:5000/#/experiments/1/runs/b5366d6e3d5e4ac6b2622cac45de9702
🧪 View experiment at: http://mlflow:5000/#/experiments/1


,run_name,macro_f1,accuracy,fit_seconds
0,random_forest_n_estimators=200_max_depth=None,0.851,0.892,2.009
1,random_forest_n_estimators=100_max_depth=None,0.850,0.891,1.059
2,random_forest_n_estimators=50_max_depth=None,0.843,0.885,0.598
3,hist_gradient_boosting_learning_rate=0.2_max_l...,0.828,0.857,3.856
4,hist_gradient_boosting_learning_rate=0.1_max_l...,0.822,0.849,4.996
5,random_forest_n_estimators=100_max_depth=20,0.815,0.857,0.984
6,random_forest_n_estimators=200_max_depth=20,0.810,0.859,1.812
7,hist_gradient_boosting_learning_rate=0.05_max_...,0.805,0.831,6.988
8,random_forest_n_estimators=50_max_depth=20,0.804,0.850,0.591
9,hist_gradient_boosting_learning_rate=0.2_max_l...,0.804,0.828,3.522


## 7. Register the best run, champion vs challenger

The best run of this sweep always becomes a new version of the registered model
`covertype` (history). The `@champion` alias moves to it only if it beats the
current champion **on the same test rows**. The API always serves
`models:/covertype@champion`.

In [8]:
client = MlflowClient()
best = results.loc[0]

new_version = mlflow.register_model(best["model_uri"], MODEL_NAME)
client.set_model_version_tag(MODEL_NAME, new_version.version, "macro_f1", f"{best['macro_f1']:.4f}")
client.set_model_version_tag(MODEL_NAME, new_version.version, "sweep", sweep_name)

try:
    champion = client.get_model_version_by_alias(MODEL_NAME, CHAMPION_ALIAS)
except MlflowException:
    champion = None

if champion is None:
    promote, reason = True, "no champion yet"
else:
    champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{CHAMPION_ALIAS}")
    champion_f1 = f1_score(y_test, champion_model.predict(X_test), average="macro")
    promote = best["macro_f1"] > champion_f1
    reason = (f"challenger v{new_version.version} {best['macro_f1']:.4f} vs "
              f"champion v{champion.version} {champion_f1:.4f}")

if promote:
    client.set_registered_model_alias(MODEL_NAME, CHAMPION_ALIAS, new_version.version)
client.set_model_version_tag(MODEL_NAME, new_version.version, "promoted", str(promote).lower())

print(f"Registered {MODEL_NAME} v{new_version.version} ({best['run_name']}).")
print(f"{'PROMOTED to @' + CHAMPION_ALIAS if promote else 'NOT promoted'}: {reason}")

Successfully registered model 'covertype'.


2026/10/04 19:43:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: covertype, version 1


Created version '1' of model 'covertype'.


Registered covertype v1 (random_forest_n_estimators=200_max_depth=None).
PROMOTED to @champion: no champion yet


## 8. Tell the inference API

Only when `@champion` moved. If the API is not running, it loads the champion
when it starts.

In [9]:
if promote:
    try:
        response = requests.post(f"{API_URL}/reload", timeout=120)
        print(response.status_code, response.json())
    except requests.ConnectionError:
        print("API not reachable: it will load the champion when it starts.")
else:
    print("Champion unchanged: no reload needed.")

API not reachable: it will load the champion when it starts.


## 9. Registered versions

In [10]:
# search_model_versions does not return aliases: fetch each version.
versions = [client.get_model_version(MODEL_NAME, v.version)
            for v in client.search_model_versions(f"name = '{MODEL_NAME}'")]
pd.DataFrame([{
    "version": int(v.version),
    "aliases": ", ".join(v.aliases),
    "macro_f1": v.tags.get("macro_f1"),
    "promoted": v.tags.get("promoted"),
    "sweep": v.tags.get("sweep"),
    "run_id": v.run_id,
} for v in versions]).sort_values("version", ignore_index=True)

,version,aliases,macro_f1,promoted,sweep,run_id
0,1,champion,0.8510,true,sweep_20261004T193736Z,ed49047f16af4176a9e57d792326f989
